<table style="width: 100%;">
<tr>
<td style="width: 50%; text-align: right; vertical-align: middle;">
<img src="https://github.com/gitpizzanow/dummy-files/blob/main/tp3nlp.jpg?raw=true" width="150">
</td>
<td style="width: 50%; text-align: left; vertical-align: middle;">

##  (NLP)   | TF-IDF + Cosine Similarity
> [SERIE](https://tp3-nlp-ing4.netlify.app/)


* *Document Frequency (DF)*
* *IDF + Smoothing*
* *TF (Term Frequency)*
* *Cosine Similarity*



</td>
</tr>
</table>

>Data: Wikipedia-like dataset

In [ ]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
).data[:3000]

In [ ]:
type(docs)

In [ ]:
docs[10]

![STEP 1 - Preprocessing](https://img.shields.io/badge/STEP%201%20-%20Preprocessing-blue)

In [ ]:
import re
import string

# Stop words list (common English stop words)
STOP_WORDS = {
    'a', 'an', 'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'is', 'are', 'was', 'were', 'be', 'been',
    'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
    'could', 'should', 'may', 'might', 'shall', 'can', 'this', 'that',
    'these', 'those', 'i', 'you', 'he', 'she', 'it', 'we', 'they',
    'me', 'him', 'her', 'us', 'them', 'my', 'your', 'his', 'its',
    'our', 'their', 'what', 'which', 'who', 'not', 'no', 'so', 'if',
    'as', 'up', 'out', 'about', 'into', 'than', 'then', 'just', 'more'
}

def preprocess(text):
    """
    Preprocesses a text document:
    1. Lowercase
    2. Remove punctuation and special characters
    3. Tokenize (split into words)
    4. Remove stop words
    5. Remove short tokens (len <= 2)
    Returns a list of clean tokens.
    """
    # 1. Lowercase
    text = text.lower()

    # 2. Remove punctuation and non-alphabetic characters
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 3. Tokenize
    tokens = text.split()

    # 4. Remove stop words and short tokens
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]

    return tokens

In [ ]:
# Test preprocess
print(preprocess(docs[10]))

![STEP 2 - Vocabulary](https://img.shields.io/badge/STEP%202%20-%20Vocabulary-blue)

In [ ]:
def build_vocab(docs):
    """
    Builds a sorted vocabulary list from all documents.
    Each document is first preprocessed, then all unique
    tokens across all documents are collected and sorted.
    Returns a sorted list of unique words (the vocabulary).
    """
    vocab_set = set()

    for doc in docs:
        tokens = preprocess(doc)
        vocab_set.update(tokens)

    # Sort alphabetically (as seen in TP2)
    vocab = sorted(list(vocab_set))
    return vocab

[![STEP 3 DF](https://img.shields.io/badge/STEP_3_DF-Document_Frequency-pink)](https://digitalpro.dev)

In [ ]:
def compute_df(docs, vocab):
    """
    Computes Document Frequency (DF) for each word in the vocabulary.
    DF(word) = number of documents that contain the word (at least once).
    Returns a dict: { word -> df_count }
    """
    df = {word: 0 for word in vocab}

    for doc in docs:
        tokens = set(preprocess(doc))  # set: count word once per document
        for word in tokens:
            if word in df:
                df[word] += 1

    return df

[![STEP 4 IDF](https://img.shields.io/badge/STEP_4_IDF-Inverse_Document_Frequency-pink)](https://digitalpro.dev)

In [ ]:
import math

def compute_idf(df, N):
    """
    Computes IDF with smoothing for each word.
    Formula (with smoothing to avoid division by zero):
        IDF(word) = log( (N + 1) / (df(word) + 1) ) + 1

    N   : total number of documents
    df  : dict { word -> document_frequency }
    Returns a dict: { word -> idf_score }
    """
    idf = {}
    for word, freq in df.items():
        idf[word] = math.log((N + 1) / (freq + 1)) + 1
    return idf

[![STEP 5 TF Vector](https://img.shields.io/badge/STEP_5_TF-Term_Frequency_Vector-pink)](https://digitalpro.dev)

In [ ]:
def compute_tf(doc, vocab):
    """
    Computes the Term Frequency (TF) vector for a single document.
    TF(word, doc) = count(word in doc) / total_words_in_doc

    Returns a list of floats (one value per word in vocab).
    """
    import numpy as np

    tokens = preprocess(doc)
    total = len(tokens) if len(tokens) > 0 else 1  # avoid division by zero

    # Count occurrences of each vocab word in the doc
    word_count = {}
    for t in tokens:
        word_count[t] = word_count.get(t, 0) + 1

    # Build the TF vector aligned with vocab order
    tf_vector = np.array([word_count.get(word, 0) / total for word in vocab])

    return tf_vector

[![STEP 6 TF-IDF Matrix](https://img.shields.io/badge/STEP_6_TFIDF-Build_TF_IDF_Matrix-pink)](https://digitalpro.dev)

In [ ]:
import numpy as np

def build_tfidf(docs):
    """
    Builds the full TF-IDF matrix for all documents.
    Steps:
      1. Build vocabulary from all docs
      2. Compute DF for each word
      3. Compute IDF (with smoothing)
      4. For each doc: compute TF vector, then multiply element-wise by IDF

    Returns:
      tfidf_matrix : numpy array of shape (num_docs, vocab_size)
      vocab        : list of words (column labels)
    """
    # Step 1 - Vocabulary
    vocab = build_vocab(docs)
    N = len(docs)

    # Step 2 - Document Frequency
    df = compute_df(docs, vocab)

    # Step 3 - IDF with smoothing
    idf = compute_idf(df, N)
    idf_vector = np.array([idf[word] for word in vocab])

    # Step 4 - Build TF-IDF matrix
    tfidf_matrix = []
    for doc in docs:
        tf_vector = compute_tf(doc, vocab)
        tfidf_vector = tf_vector * idf_vector  # element-wise multiplication
        tfidf_matrix.append(tfidf_vector)

    tfidf_matrix = np.array(tfidf_matrix)
    return tfidf_matrix, vocab

[![STEP 7 Cosine Similarity](https://img.shields.io/badge/STEP_7-Cosine_Similarity-pink)](https://digitalpro.dev)

In [ ]:
def cosine(a, b):
    """
    Computes the Cosine Similarity between two vectors a and b.

    Formula:
        cosine(a, b) = dot(a, b) / (||a|| * ||b||)

    Returns a float between 0 and 1.
    Returns 0.0 if either vector is a zero vector (no division by zero).
    """
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot_product / (norm_a * norm_b)

[![STEP 8 Search Engine](https://img.shields.io/badge/STEP_8-Search_Engine_REAL_VERSION-orange)](https://digitalpro.dev)

In [ ]:
import numpy as np
def search(query, docs, top_k=5):
    tfidf_matrix, vocab = build_tfidf(docs)

    q_vec = np.zeros(len(vocab)) # build query vector
    q_words = preprocess(query)

    for i, w in enumerate(vocab):
        q_vec[i] = q_words.count(w)

    if np.sum(q_vec) > 0:
        q_vec = q_vec / np.sum(q_vec)

    scores = []

    for i, doc_vec in enumerate(tfidf_matrix):
        score = cosine(q_vec, doc_vec)
        scores.append((score, i))

    scores.sort(reverse=True)

    return scores[:top_k]

> TEST

In [ ]:
query = "machine learning neural network"
results = search(query, docs)

for score, idx in results:
    print(score)
    print(docs[idx][:200])
    print("-" * 50)